<a href="https://colab.research.google.com/github/Longhanhmid24/DoAn_Gen_Images/blob/main/stable_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q git+https://github.com/huggingface/diffusers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
!pip install -q diffusers transformers accelerate peft xformers datasets
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00


In [29]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py

In [30]:
import pandas as pd
import json

csv_path = "/content/drive/MyDrive/Computer_Vision/train/captions_ready_to_train.csv"
output_path = "/content/drive/MyDrive/Computer_Vision/train/metadata.jsonl"

df = pd.read_csv(csv_path)

with open(output_path, "w") as f:
    for _, row in df.iterrows():
        item = {
            "file_name": f"Images/{row['image']}",
            "text": row['caption']
        }
        f.write(json.dumps(item) + "\n")

print("Done!")

Done!


In [32]:
import os
import json

jsonl_path = "/content/drive/MyDrive/Computer_Vision/train/metadata.jsonl"
base_dir = "/content/drive/MyDrive/Computer_Vision/train"

valid_lines = []
count_ok = 0
count_bad = 0

with open(jsonl_path, "r") as f:
    for line in f:
        item = json.loads(line)
        img_path = os.path.join(base_dir, item["file_name"])

        if os.path.exists(img_path):
            valid_lines.append(item)
            count_ok += 1
        else:
            count_bad += 1

# Ghi đè lại file cũ
with open(jsonl_path, "w") as f:
    for item in valid_lines:
        f.write(json.dumps(item) + "\n")

print("Kept:", count_ok)
print("Removed:", count_bad)

Kept: 78000
Removed: 1519


#### Huấn luyện mô hình Stable Diffusion XL

In [35]:
!accelerate launch \
  --mixed_precision="bf16" \
  --num_processes=1 \
  --num_machines=1 \
  --dynamo_backend="no" \
  train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/content/drive/MyDrive/Computer_Vision/train" \
  --resolution=1024 \
  --train_batch_size=8 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=50 \
  --max_grad_norm=1.0 \
  --max_train_steps=2000 \
  --checkpointing_steps=1500 \
  --output_dir="/content/drive/MyDrive/Computer_Vision/train/lora-output" \
  --logging_dir="/content/drive/MyDrive/Computer_Vision/logs" \
  --report_to="tensorboard" \
  --enable_xformers_memory_efficient_attention \
  --gradient_checkpointing \
  --seed=42

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
{'clip_sample_range', 'rescale_betas_zero_snr', 'variance_type', 'dynamic_thresholding_ratio', 'thresholding'} was not found in 

#### Đánh giá mô hình

In [ ]:
!pip install pytorch-fid torchmetrics matplotlib torchvision